# 交叉验证：最可靠的模型评估方法
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：12_模型评估与选择 | 源文件：交叉验证.py | 核心功能：K-Fold、分层 K-Fold、交叉验证评分

## 概述

交叉验证将数据分成 K 份，轮流用其中 K-1 份训练、1 份验证，取 K 次结果的平均。比单次 train/test split 更稳定可靠。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    KFold, StratifiedKFold, cross_val_score, cross_validate, learning_curve
)
# --- 导入库 ---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 生成数据
X, y = make_classification(
    n_samples=300, n_features=10, n_informative=5,
    n_classes=3, n_clusters_per_class=1, random_state=42
)

model = LogisticRegression(max_iter=1000, random_state=42)

## 数学原理

### 1. K 折交叉验证

将数据集 $D$ 均匀划分为 $K$ 个不重叠的子集 $D_1, D_2, \ldots, D_K$。第 $k$ 轮用 $D_k$ 做验证集，其余做训练集：

$$\text{CV}_k = \frac{1}{|D_k|}\sum_{(x_i, y_i) \in D_k} L(y_i, f_{-k}(x_i))$$

最终交叉验证估计：

$$\text{CV} = \frac{1}{K}\sum_{k=1}^{K} \text{CV}_k$$

### 2. 偏差-方差权衡

- $K$ 较小（如 3）：训练集少，偏差较大，方差较小
- $K$ 较大（如 10）：训练集接近全集，偏差较小，方差较大
- $K=n$（留一法 LOO）：偏差最小，方差最大，计算量最大

$$\text{Var}[\text{CV}] \propto \frac{1}{K}$$

### 3. 分层交叉验证（StratifiedKFold）

分类任务中，保持每个折的类别比例与整体一致：

$$\frac{|y=k \cap D_j|}{|D_j|} \approx \frac{|y=k|}{n}, \quad \forall k, j$$

避免某个折中缺少某个类别。

### 4. 交叉验证的无偏性

$K$ 折 CV 是泛化误差的近似无偏估计：

$$\mathbb{E}[\text{CV}] \approx \mathbb{E}[L(y, f_{D}(x))]$$

但存在小的悲观偏差（训练集比全集小）。

### 5. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `KFold(n_splits=5)` | $K=5$，数据分为 5 个子集 |
| `StratifiedKFold(n_splits=5)` | 分层 $K$ 折，保持类别比例 |
| `cross_val_score(model, X, y, cv=5)` | 返回 $[\text{CV}_1, \ldots, \text{CV}_5]$ |
| `cross_validate(model, X, y, cv=5, return_train_score=True)` | 同时返回训练和验证分数 |
| `learning_curve(model, X, y, train_sizes=...)` | 不同训练集大小下的性能曲线 |

### 1. KFold 基础K折交叉验证

运行 1. KFold 基础K折交叉验证 的代码，观察算法在该环节的行为。

In [ ]:
print("=" * 60)
print("1. KFold 基础K折交叉验证")
print("=" * 60)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print(f"数据总量: {len(X)}, K={kf.n_splits}")
print()

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"第{fold_idx+1}折: 训练集={len(train_idx)}样本, 验证集={len(val_idx)}样本")
    print(f"  训练集标签分布: {np.bincount(y[train_idx])}")
    print(f"  验证集标签分布: {np.bincount(y[val_idx])}")

### 2. StratifiedKFold 分层K折

运行 2. StratifiedKFold 分层K折 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("2. StratifiedKFold 分层K折")
print("=" * 60)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("分层K折保持每折的类别比例与整体一致:")
print(f"整体类别比例: {np.bincount(y) / len(y)}")
print()

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"第{fold_idx+1}折: 训练集={len(train_idx)}, 验证集={len(val_idx)}")
    print(f"  训练集类别比例: {np.round(np.bincount(y[train_idx]) / len(train_idx), 3)}")
    print(f"  验证集类别比例: {np.round(np.bincount(y[val_idx]) / len(val_idx), 3)}")

### 3. cross_val_score 快速交叉验证

运行 3. cross_val_score 快速交叉验证 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("3. cross_val_score 快速交叉验证")
print("=" * 60)

# 单指标交叉验证
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"5折交叉验证准确率: {scores}")
print(f"平均准确率: {scores.mean():.4f} (+/- {scores.std():.4f})")

# 多种指标
for metric in ['accuracy', 'f1_weighted', 'precision_weighted']:
    scores_m = cross_val_score(model, X, y, cv=5, scoring=metric)
    print(f"{metric}: {scores_m.mean():.4f} (+/- {scores_m.std():.4f})")

### 4. cross_validate 多指标评估

在测试集上评估模型性能，关注关键指标。

In [ ]:
print("\n" + "=" * 60)
print("4. cross_validate 多指标 + 训练/测试时间")
print("=" * 60)

scoring = {
    'accuracy': 'accuracy',
    'f1': 'f1_weighted',
    'precision': 'precision_weighted',
    'recall': 'recall_weighted',
# --- 继续 ---
}

cv_results = cross_validate(
    model, X, y, cv=5, scoring=scoring,
    return_train_score=True, return_estimator=True
)

print(f"{'指标':<25} {'训练集均值':>10} {'验证集均值':>10} {'验证集标准差':>12}")
print("-" * 60)
for key in scoring:
    train_key = f'train_{key}'
    test_key = f'test_{key}'
# --- 条件判断 ---
    if train_key in cv_results and test_key in cv_results:
        train_mean = cv_results[train_key].mean()
        test_mean = cv_results[test_key].mean()
        test_std = cv_results[test_key].std()
        print(f"  {key:<23} {train_mean:>10.4f} {test_mean:>10.4f} {test_std:>12.4f}")

print(f"\n训练时间 (5折总计): {cv_results['fit_time'].sum():.3f}秒")
print(f"预测时间 (5折总计): {cv_results['score_time'].sum():.3f}秒")

### 5. 不同模型的交叉验证对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n" + "=" * 60)
print("5. 不同模型的交叉验证对比")
print("=" * 60)

models = {
    '逻辑回归': LogisticRegression(max_iter=1000, random_state=42),
    '随机森林(10棵树)': RandomForestClassifier(n_estimators=10, random_state=42),
    '随机森林(100棵树)': RandomForestClassifier(n_estimators=100, random_state=42),
}

print(f"{'模型':<20} {'准确率均值':>10} {'准确率标准差':>12} {'F1均值':>10}")
print("-" * 55)
for name, m in models.items():
    acc_scores = cross_val_score(m, X, y, cv=5, scoring='accuracy')
    f1_scores = cross_val_score(m, X, y, cv=5, scoring='f1_weighted')
# --- 输出结果 ---
    print(f"  {name:<18} {acc_scores.mean():>10.4f} {acc_scores.std():>12.4f} {f1_scores.mean():>10.4f}")

### 6. 学习曲线

运行 6. 学习曲线 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("6. learning_curve 学习曲线")
print("=" * 60)

train_sizes, train_scores, val_scores = learning_curve(
    model, X, y,
    train_sizes=np.linspace(0.1, 1.0, 5),
    cv=5, scoring='accuracy', random_state=42
)

print(f"{'训练集大小':>10} {'训练集准确率':>12} {'验证集准确率':>12}")
print("-" * 38)
for size, train_mean, val_mean in zip(
    train_sizes,
    train_scores.mean(axis=1),
# --- 继续 ---
    val_scores.mean(axis=1)
):
    print(f"  {size:>8} {train_mean:>12.4f} {val_mean:>12.4f}")

print("\n分析:")
gap = train_scores.mean(axis=1)[-1] - val_scores.mean(axis=1)[-1]
print(f"  最终训练-验证差距: {gap:.4f}")
if gap > 0.05:
    print("  -> 存在过拟合迹象 (训练集远高于验证集)")
# --- 条件判断 ---
elif val_scores.mean(axis=1)[-1] < 0.6:
    print("  -> 可能欠拟合 (整体准确率偏低)")
else:
    print("  -> 模型表现良好")

## 关键代码解释

```python
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print(f"平均准确率: {scores.mean():.4f} +/- {scores.std():.4f}")
```

## 注意事项

1. 分类任务用 `StratifiedKFold`
2. 时间序列用 `TimeSeriesSplit`
3. 交叉验证不应用于最终报告（应用独立测试集）

## 总结与延伸

以上代码展示了 **交叉验证** 的完整流程。

**解读要点**：
- 关注 **ROC 曲线** 下的面积（AUC）
- 观察 **交叉验证** 各折的方差
- 对比不同评估指标的适用场景

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **嵌套交叉验证**：外层评估、内层调参
- **Leave-One-Out**：极小数据集
- **分组交叉验证**：同组数据不同时出现在训练和验证集
